# Variant A — fully nested participant-level evaluation

This notebook evaluates **only Variant A**: fold-local LPGNet linear-prediction residuals,
squeeze–excitation after each convolutional block, and attention pooling. The corrected
LPGNet baseline is evaluated in notebook 1, and the GRU-residual variant has been removed.
No baseline or GRU model is trained here.

For each declared training seed and outer fold:

1. Build participant-disjoint inner folds using outer-training participants only.
2. Evaluate 12 candidates on the same inner folds with fold-local LP fitting.
3. Select by pooled inner participant log loss, with AUC as a tie-breaker.
4. Re-estimate early-stopping epochs using held-out outer-training participants.
5. Refit both stages on all outer-training participants for the selected epochs.
6. Predict outer-test recordings once and store exact OOF probabilities.

Participant-level OOF metrics are primary. Recording probabilities are averaged within each
participant, the decision threshold remains 0.5, and repeated seeds are not additional
participants. Window and recording arrays are generated batch by batch. Completed A outer
folds are checkpointed under `lpgnet_variant_a_outputs` and skipped after a kernel restart.


In [ ]:
# ============================================================
# Reproducibility and experiment settings
# ============================================================
import os, time, random, json, urllib.request, zipfile, warnings, gc, ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import lfilter
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             log_loss, roc_curve)
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH','true')  # must precede TF initialization
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 1                 # official LPGNet notebook seed
FOLD_SEED = 0            # official StratifiedKFold seed
N_FOLDS = 10
N_INNER = 3
VAL_FRAC = 0.15
LP_ORDER = 11
WIN, STRIDE = 100, 50
BATCH_WIN, BATCH_REC = 128, 64
LR_WIN, LR_REC = 5e-4, 1e-3
LABEL_SMOOTH = 0.1
L2_REG = 1e-3
MAX_EPOCHS_WIN, MAX_EPOCHS_REC = 1000, 200
PATIENCE, RLR_PATIENCE, MIN_LR = 25, 14, 1e-7
N_BOOT = 1000
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# ------------------------ publication protocol knobs ------------------------
TRAIN_SEEDS = [42]          # use multiple declared seeds for final robustness analyses
N_TRIALS = 12               # candidates per outer fold per seed; unchanged
N_BOOT = 1000
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_DIR = Path('lpgnet_variant_a_outputs')


In [ ]:
# ============================================================
# Download PhysioNet data and attach verified clinical metadata
# ============================================================
URL = 'https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip'
ZIP = 'gaitpdb.zip'; DATA_DIR = 'gaitpdb_data'
ROOT = os.path.join(DATA_DIR, 'gait-in-parkinsons-disease-1.0.0')
if not os.path.isdir(ROOT):
    if not os.path.exists(ZIP):
        print('Downloading PhysioNet gaitpdb (~300 MB, one-off)...')
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)

records = []
for fn in sorted(os.listdir(ROOT)):
    if not fn.endswith('.txt') or fn[2:4] not in ('Co', 'Pt'):
        continue
    base = fn[:-4]
    arr = np.loadtxt(os.path.join(ROOT, fn), delimiter='\t')
    records.append(dict(file=fn, study=base[:2], subject=base.split('_')[0],
                        label=int(base[2:4] == 'Pt'),
                        signal=arr[:, 1:19].astype(np.float32)))
subjects = sorted({r['subject'] for r in records})
subj_label = {s: next(r['label'] for r in records if r['subject'] == s) for s in subjects}
by_file = {r['file']: i for i, r in enumerate(records)}
by_subject = {s: [i for i, r in enumerate(records) if r['subject'] == s] for s in subjects}
print(f"{len(records)} recordings | {len(subjects)} participants with sensor files | "
      f"{sum(subj_label.values())} PD / {len(subjects)-sum(subj_label.values())} controls")

# demographics.txt is distributed with gaitpdb. Missing values remain missing.
demo_path = os.path.join(ROOT, 'demographics.txt')
demo = pd.read_csv(demo_path, sep='\t', engine='python', usecols=range(20))
demo = demo.loc[:, ~demo.columns.str.startswith('Unnamed')].copy()
demo = demo.dropna(subset=['ID']).copy()
demo['ID'] = demo['ID'].astype(str)
demo = demo.drop_duplicates('ID').set_index('ID')
clinical = {}
for s in subjects:
    row = demo.loc[s] if s in demo.index else None
    hy = pd.to_numeric(row.get('HoehnYahr'), errors='coerce') if row is not None else np.nan
    # Controls are not assigned HY=0 by inference. Only recorded PD values are used.
    verified_hy = float(hy) if subj_label[s] == 1 and pd.notna(hy) else np.nan
    clinical[s] = dict(
        study=s[:2], diagnosis='PD' if subj_label[s] else 'Control',
        hy=verified_hy, hy_label=(f'{verified_hy:g}' if pd.notna(verified_hy)
                                  else ('Unknown' if subj_label[s] else 'Not applicable')),
        hy_status=('verified' if pd.notna(verified_hy) else
                   ('missing' if subj_label[s] else 'not_applicable')),
        age=pd.to_numeric(row.get('Age'), errors='coerce') if row is not None else np.nan,
        gender=pd.to_numeric(row.get('Gender'), errors='coerce') if row is not None else np.nan)
missing_metadata = [s for s in subjects if s not in demo.index]
print('Participants without a demographics join:', missing_metadata)


In [ ]:
# ============================================================
# Official preprocessing and LP residuals
# ============================================================
def preprocess(x):
    # Causal two-tap moving average, then decimation 100 -> 50 Hz.
    return lfilter([0.5, 0.5], 1, x, axis=0)[::2]

def fit_lp(control_sigs, order=LP_ORDER):
    """One predictor per channel; MATLAB-compatible pooled-control fitting."""
    C, conc_all = np.zeros((18, order)), []
    for c in range(18):
        # generateLPresidual.m starts with zero padding and inserts it between walks.
        pieces = [np.zeros(order, np.float32)]
        pieces += [np.concatenate([s[:, c], np.zeros(order, np.float32)])
                   for s in control_sigs]
        conc = np.concatenate(pieces)
        conc_all.append(conc)
        n = len(conc)
        nfft = 1 << (2*n - 1).bit_length()
        f = np.fft.rfft(conc, nfft)
        r = np.fft.irfft(f*np.conj(f), nfft)[:order+1] / n
        R = np.array([[r[abs(i-j)] for j in range(order)] for i in range(order)])
        C[c] = np.linalg.solve(R, r[1:order+1])
    # MATLAB std uses N-1 normalization. No test recordings enter this calculation.
    stds = np.std(np.stack(conc_all, axis=1), axis=0, ddof=1) + 1e-8
    return C, stds

def apply_lp(x, C, stds):
    e = np.stack([lfilter(np.r_[1.0, -C[c]], 1, x[:, c]) for c in range(18)], axis=1)
    return (e / stds).astype(np.float32)

def lp_features(indices, C, stds):
    return {i: apply_lp(records[i]['proc'], C, stds) for i in indices}

for r in records:
    r['proc'] = preprocess(r['signal'])


In [ ]:
# ============================================================
# Fixed participant-disjoint outer folds and assertions
# ============================================================
def make_folds():
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=FOLD_SEED)
    arr = np.array(subjects); y = np.array([subj_label[s] for s in subjects])
    folds = []
    for tr_s, te_s in skf.split(arr, y):
        tr_subj, te_subj = set(arr[tr_s]), set(arr[te_s])
        assert not tr_subj & te_subj
        tr = [i for i, r in enumerate(records) if r['subject'] in tr_subj]
        te = [i for i, r in enumerate(records) if r['subject'] in te_subj]
        assert not ({records[i]['subject'] for i in tr} & {records[i]['subject'] for i in te})
        folds.append(dict(train=tr, test=te))
    return folds

def split_val(indices, frac=VAL_FRAC, seed=SEED):
    """Split participants, never recordings; output participant sets are disjoint."""
    ss = sorted({records[i]['subject'] for i in indices})
    yy = [subj_label[s] for s in ss]
    if len(set(yy)) > 1:
        splitter = StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
        tr_s, va_s = next(splitter.split(np.zeros(len(ss)), yy))
    else:
        perm = np.random.default_rng(seed).permutation(len(ss))
        n_va = max(1, int(round(frac*len(ss))))
        va_s, tr_s = perm[:n_va], perm[n_va:]
    tr_subj, va_subj = {ss[i] for i in tr_s}, {ss[i] for i in va_s}
    assert not tr_subj & va_subj
    return ([i for i in indices if records[i]['subject'] in tr_subj],
            [i for i in indices if records[i]['subject'] in va_subj])

def labels(indices):
    return np.array([records[i]['label'] for i in indices], dtype=np.float32)

FOLDS = make_folds()
for k, f in enumerate(FOLDS):
    tr_n = len({records[i]['subject'] for i in f['train']})
    te_n = len({records[i]['subject'] for i in f['test']})
    print(f"Fold {k}: {tr_n} train participants / {te_n} test participants | "
          f"{len(f['train'])} train / {len(f['test'])} test recordings")


In [ ]:
# ============================================================
# Variant-A-compatible recording training and bounded-memory batches
# ============================================================
def official_sepconv(filters, kernel_size, reg):
    """Use the released call; provide an equivalent modern-Keras fallback."""
    try:
        return layers.SeparableConv1D(filters, kernel_size, activation='linear',
                                      padding='valid', kernel_regularizer=reg)
    except (TypeError, ValueError):
        # Keras 3 raises ValueError; older versions may raise TypeError.
        # Modern Keras exposes separate depthwise/pointwise regularizer arguments.
        return layers.SeparableConv1D(filters, kernel_size, activation='linear',
                                      padding='valid', depthwise_regularizer=reg,
                                      pointwise_regularizer=reg)

def build_recording_model(window_model):
    """Official-equivalent stage 2: frozen pooled backbone, new dropout + sigmoid dense layer."""
    backbone = keras.Model(window_model.input, window_model.get_layer('pooled').output)
    backbone.trainable = False
    inp = layers.Input((None, 18))
    x = backbone(inp)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out)

def callbacks():
    return [keras.callbacks.EarlyStopping('val_loss', patience=PATIENCE,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.25,
                                              patience=RLR_PATIENCE, min_lr=MIN_LR)]

class WindowSequence(keras.utils.Sequence):
    """Materialize only one window batch; preserve the official endpoint convention."""
    def __init__(self,indices,feats,batch_size=BATCH_WIN,shuffle=True,seed=SEED):
        super().__init__(); self.feats=feats; self.batch_size=batch_size; self.shuffle=shuffle
        self.pairs=[(i,st) for i in indices for st in range(0,len(feats[i])-WIN,STRIDE)]
        self.rng=np.random.default_rng(seed); self.indices=np.arange(len(self.pairs))
        if self.shuffle: self.rng.shuffle(self.indices)
    def __len__(self): return int(np.ceil(len(self.pairs)/self.batch_size))
    def __getitem__(self,j):
        ids=self.indices[j*self.batch_size:(j+1)*self.batch_size]
        pairs=[self.pairs[int(k)] for k in ids]
        X=np.stack([self.feats[i][st:st+WIN] for i,st in pairs]).astype(np.float32)
        y=np.array([records[i]['label'] for i,st in pairs],dtype=np.float32)
        return X,y
    def on_epoch_end(self):
        if self.shuffle: self.rng.shuffle(self.indices)

class RecordingSequence(keras.utils.Sequence):
    """Wrap-pad batches to the same sequence-wide maximum as the original pad_wrap."""
    def __init__(self,indices,feats,batch_size=BATCH_REC,shuffle=False,seed=SEED):
        super().__init__(); self.indices=list(indices); self.feats=feats
        self.batch_size=batch_size; self.shuffle=shuffle; self.rng=np.random.default_rng(seed)
        self.L=max(len(feats[i]) for i in self.indices); self.order=np.arange(len(self.indices))
        if self.shuffle: self.rng.shuffle(self.order)
    def __len__(self): return int(np.ceil(len(self.indices)/self.batch_size))
    def __getitem__(self,j):
        ids=self.order[j*self.batch_size:(j+1)*self.batch_size]
        batch=[self.indices[int(k)] for k in ids]
        X=np.stack([np.pad(self.feats[i],((0,self.L-len(self.feats[i])),(0,0)),mode='wrap')
                    for i in batch]).astype(np.float32)
        return X,labels(batch)
    def on_epoch_end(self):
        if self.shuffle: self.rng.shuffle(self.order)

def fit_options():
    # Keras 3's TensorFlow trainer manages Sequence execution directly.
    # workers/use_multiprocessing/max_queue_size are unsupported here.
    return {}

def release_memory():
    keras.backend.clear_session(); gc.collect()

def fit_two_stage(builder,cfg,tr,va,feats,max_win=MAX_EPOCHS_WIN,
                  max_rec=MAX_EPOCHS_REC,verbose=0):
    """Official optimization with bounded-memory batches and participant-disjoint tr/va."""
    model=builder(**cfg) if cfg else builder()
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN,clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'])
    h1=model.fit(WindowSequence(tr,feats,shuffle=True,seed=SEED),epochs=max_win,
                 validation_data=RecordingSequence(va,feats,shuffle=False),
                 callbacks=callbacks(),verbose=verbose,**fit_options())
    full=build_recording_model(model)
    full.compile(optimizer=keras.optimizers.Adam(LR_REC,clipvalue=1),
                 loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=['accuracy'])
    h2=full.fit(RecordingSequence(tr,feats,shuffle=True,seed=SEED+1),epochs=max_rec,
                validation_data=RecordingSequence(va,feats,shuffle=False),
                callbacks=callbacks(),verbose=verbose,**fit_options())
    best=dict(window=int(np.argmin(h1.history['val_loss'])+1),
              recording=int(np.argmin(h2.history['val_loss'])+1))
    return model,full,h1.history,h2.history,best

def refit_two_stage(builder,cfg,train_idx,feats,epochs,verbose=0):
    """Refit on every outer-training participant without outer-test data."""
    model=builder(**cfg) if cfg else builder()
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN,clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'])
    model.fit(WindowSequence(train_idx,feats,shuffle=True,seed=SEED),
              epochs=max(1,epochs['window']),verbose=verbose,**fit_options())
    full=build_recording_model(model)
    full.compile(optimizer=keras.optimizers.Adam(LR_REC,clipvalue=1),
                 loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                 metrics=['accuracy'])
    full.fit(RecordingSequence(train_idx,feats,shuffle=True,seed=SEED+1),
             epochs=max(1,epochs['recording']),verbose=verbose,**fit_options())
    return model,full

def predict_recordings(full,indices,feats):
    return np.array([float(full.predict(feats[i][None],verbose=0)[0,0]) for i in indices])

In [ ]:
# ============================================================
# OOF metrics, participant aggregation, errors, and clinical subgroups
# ============================================================
def metric_dict(y, p):
    y = np.asarray(y, int); p = np.asarray(p, float); yh = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    return dict(n=len(y), accuracy=accuracy_score(y, yh),
                balanced_accuracy=balanced_accuracy_score(y, yh),
                macro_f1=f1_score(y, yh, average='macro', zero_division=0),
                f1_pd=f1_score(y, yh, zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=log_loss(y, np.clip(p,1e-7,1-1e-7), labels=[0,1]))

def oof_frame(rows):
    df = pd.DataFrame(rows)
    df['prediction'] = (df['probability'] >= 0.5).astype(int)
    df['correct'] = (df['prediction'] == df['label']).astype(int)
    df['confidence'] = np.maximum(df['probability'], 1-df['probability'])
    df['nll'] = -(df['label']*np.log(np.clip(df['probability'],1e-7,1)) +
                  (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col] = df['subject'].map(lambda s: clinical[s][col])
    return df

def participant_oof(df, group_cols):
    """Equal weight per participant; probability is averaged over that person's walks."""
    keys = group_cols + ['subject']
    agg = dict(label=('label','first'), fold=('fold','first'), probability=('probability','mean'),
               recordings=('file','nunique'))
    out = df.groupby(keys, as_index=False).agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        rec = metric_dict(g['label'], g['probability'])
        part = participant_oof(g, group_cols)
        pm = metric_dict(part['label'], part['probability'])
        rows.append({**base, **{'recording_'+k:v for k,v in rec.items()}})
        rows[-1].update({'participant_'+k:v for k,v in pm.items()})
    return pd.DataFrame(rows)

def wilson_interval(correct, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = correct/n; den = 1+z*z/n
    center = (ph+z*z/(2*n))/den
    half = z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half, center+half

def error_tables(df, group_cols):
    part = participant_oof(df, group_cols)
    errors = part[part['correct'] == 0].sort_values(group_cols+['nll'], ascending=[True]*len(group_cols)+[False])
    pd_rows = []
    for keys, g in part[part['label'] == 1].groupby(group_cols+['hy_label'], dropna=False):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['hy_label'], keys))
        n = len(g); correct = int(g['correct'].sum()); lo, hi = wilson_interval(correct, n)
        pd_rows.append({**base, 'pd_participants':n, 'correct':correct, 'errors':n-correct,
                        'pd_sensitivity':correct/n, 'wilson_95_low':lo, 'wilson_95_high':hi})
    study_rows = []
    for keys, g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols+['study','diagnosis'], keys))
        study_rows.append({**base, **metric_dict(g['label'], g['probability'])})
    return part, errors, pd.DataFrame(pd_rows), pd.DataFrame(study_rows)

def show_table(df, title, limit=100):
    print('\n'+title)
    display(df.head(limit))

def pooled_bootstrap(df, group_cols, metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity','roc_auc')):
    """Stratified participant bootstrap conditional on fitted OOF predictions."""
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        base = dict(zip(group_cols, keys))
        part = participant_oof(g, group_cols).reset_index(drop=True)
        y = part['label'].to_numpy(); p = part['probability'].to_numpy()
        class_idx = [np.flatnonzero(y == c) for c in (0,1)]
        rng = np.random.default_rng(2026)
        for metric in metrics:
            observed = metric_dict(y,p)[metric]
            vals = []
            for _ in range(N_BOOT):
                ii = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_idx])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**base,'metric':metric,'estimate':observed,
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_confusion_and_roc(df, group_cols, title_col):
    part = participant_oof(df, group_cols)
    groups = list(part.groupby(group_cols))
    fig, axes = plt.subplots(len(groups), 2, figsize=(11,4.3*len(groups)), squeeze=False)
    for row, (keys,g) in enumerate(groups):
        if not isinstance(keys, tuple): keys=(keys,)
        title = ' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm = confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_ = roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
    fig.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Variant A definition, search space, and fold-local LP features
# ============================================================
def se_block(x, ratio):
    ch = x.shape[-1]
    s = layers.GlobalAveragePooling1D()(x)
    s = layers.Dense(max(ch//ratio,4),activation='relu')(s)
    s = layers.Dense(ch,activation='sigmoid')(s)
    return layers.Multiply()([x,layers.Reshape((1,ch))(s)])

def attn_pool(x):
    w = layers.Softmax(axis=1)(layers.Dense(1)(x))
    return layers.Lambda(lambda t: tf.reduce_sum(t[0]*t[1],axis=1),name='pooled')([x,w])

def build_variantA(n_ch=18, channels=32, kernels=(7,5,3), se_ratio=8,
                   spat_drop=0.2, drop=0.25, l2=L2_REG, noise=0.01):
    reg=keras.regularizers.l2(l2)
    if isinstance(channels,int): channels=[channels]*(len(kernels)-1)+[2*channels]
    inp=layers.Input((None,n_ch)); x=layers.SpatialDropout1D(spat_drop)(inp)
    x=layers.GaussianNoise(noise)(x)
    for i,(k,c) in enumerate(zip(kernels,channels)):
        x=official_sepconv(c,k,reg)(x); x=layers.BatchNormalization()(x)
        x=layers.Activation('elu')(x); x=se_block(x,se_ratio)
        if i<len(kernels)-1: x=layers.MaxPool1D(2,2)(x)
    x=attn_pool(x); x=layers.Dropout(drop)(x)
    out=layers.Dense(1,activation='sigmoid',kernel_regularizer=reg)(x)
    return keras.Model(inp,out)

SPACE_A=dict(channels=[32,48],kernels=[(7,5,3),(9,7,5)],se_ratio=[4,8],
             spat_drop=[0.1,0.2,0.3],drop=[0.25,0.4],l2=[1e-4,1e-3])

def prepare_fold(train_idx,eval_idx,seed=SEED):
    """Fit order-11 LP on training controls only; evaluation data are transformed only."""
    C,stds=fit_lp([records[i]['proc'] for i in train_idx if records[i]['label']==0])
    return lp_features(train_idx+eval_idx,C,stds)

def parameter_report(cfg):
    classifier=build_variantA(**cfg)
    n_classifier=int(classifier.count_params())
    n_predictor=18*LP_ORDER
    release_memory()
    return n_classifier,n_predictor,n_classifier+n_predictor


In [ ]:
# ============================================================
# Fully nested Variant-A search: no outer-test participant enters any decision
# ============================================================
def sample_cfg(space,rng):
    return {k:v[int(rng.integers(len(v)))] for k,v in space.items()}

def inner_splits(outer_train,seed):
    ss=sorted({records[i]['subject'] for i in outer_train}); yy=[subj_label[s] for s in ss]
    skf=StratifiedKFold(N_INNER,shuffle=True,random_state=seed); splits=[]
    for tr_s,va_s in skf.split(np.zeros(len(ss)),yy):
        tr_subj={ss[i] for i in tr_s}; va_subj={ss[i] for i in va_s}
        assert not tr_subj&va_subj and va_subj<={records[i]['subject'] for i in outer_train}
        tr=[i for i in outer_train if records[i]['subject'] in tr_subj]
        va=[i for i in outer_train if records[i]['subject'] in va_subj]
        splits.append((tr,va))
    return splits

def prediction_rows(indices,p,**base):
    return [dict(**base,file=records[i]['file'],subject=records[i]['subject'],
                 label=records[i]['label'],probability=float(prob)) for i,prob in zip(indices,p)]

def evaluate_inner_candidate(cfg,splits,seed,outer_fold,trial,feature_cache):
    rows=[]; epoch_rows=[]; start=time.time()
    for j,(tr,va) in enumerate(splits):
        if j not in feature_cache: feature_cache[j]=prepare_fold(tr,va,seed=seed+100*j)
        feats=feature_cache[j]
        core,early=split_val(tr,seed=seed+1000+j)
        release_memory(); tf.keras.utils.set_random_seed(seed+10000+j)
        model,full,h1,h2,best=fit_two_stage(build_variantA,cfg,core,early,feats)
        p=predict_recordings(full,va,feats)
        rows+=prediction_rows(va,p,family='A',seed=seed,outer_fold=outer_fold,
                              trial=trial,inner_fold=j,fold=j)
        epoch_rows.append(best)
        del model,full,h1,h2,p
        release_memory()
    inner=oof_frame(rows)
    part=participant_oof(inner,['family','seed','outer_fold','trial'])
    metrics=metric_dict(part['label'],part['probability'])
    return metrics,inner,epoch_rows,time.time()-start

def nested_search(space,outer_fold,outer_train,seed):
    # Identical to the former Variant-A random stream and common inner splits.
    rng=np.random.default_rng(seed+50000*outer_fold)
    splits=inner_splits(outer_train,seed+777*outer_fold)
    trials=[]; selected_inner=[]; best_idx=None; best_key=(np.inf,-np.inf)
    feature_cache={}
    for t in range(N_TRIALS):
        cfg=sample_cfg(space,rng)
        metrics,inner,epochs,seconds=evaluate_inner_candidate(
            cfg,splits,seed,outer_fold,t,feature_cache)
        ncl,npred,ntotal=parameter_report(cfg)
        row=dict(family='A',seed=seed,outer_fold=outer_fold,trial=t,candidate=cfg,
                 inner_participant_log_loss=metrics['log_loss'],
                 inner_participant_auc=metrics['roc_auc'],
                 inner_participant_accuracy=metrics['accuracy'],
                 mean_best_window_epoch=float(np.mean([e['window'] for e in epochs])),
                 mean_best_recording_epoch=float(np.mean([e['recording'] for e in epochs])),
                 classifier_parameters=ncl,predictor_parameters=npred,total_parameters=ntotal,
                 training_seconds=seconds)
        trials.append(row); key=(metrics['log_loss'],-metrics['roc_auc'])
        if key<best_key: best_key=key; best_idx=t; selected_inner=inner
        print(f"[A] outer {outer_fold} seed {seed} trial {t:02d} | "
              f"inner NLL={metrics['log_loss']:.4f} AUC={metrics['roc_auc']:.3f} | {cfg}")
        release_memory()
    feature_cache.clear(); release_memory()
    return best_idx,trials,selected_inner

def fit_selected_outer(cfg,outer_fold,seed):
    f=FOLDS[outer_fold]; tr_all,te=f['train'],f['test']
    feats=prepare_fold(tr_all,te,seed=seed+200000)
    core,early=split_val(tr_all,seed=seed+300000)
    release_memory(); tf.keras.utils.set_random_seed(seed+400000)
    early_window,early_full,h1,h2,best=fit_two_stage(build_variantA,cfg,core,early,feats)
    del early_window,early_full,h1,h2
    release_memory(); tf.keras.utils.set_random_seed(seed+400000)
    model,full=refit_two_stage(build_variantA,cfg,tr_all,feats,best)
    p=predict_recordings(full,te,feats)
    del model,full,feats
    release_memory()
    return p,best


In [ ]:
# ============================================================
# Restart-safe Variant-A-only run; no baseline or GRU execution
# ============================================================
CHECKPOINT_DIR.mkdir(exist_ok=True)
ck_oof=CHECKPOINT_DIR/'checkpoint_recording_oof.csv'
ck_trials=CHECKPOINT_DIR/'checkpoint_search_trials.csv'
ck_sel=CHECKPOINT_DIR/'checkpoint_selected_configurations.csv'
ck_inner=CHECKPOINT_DIR/'checkpoint_selected_inner_oof.csv'

def parse_candidate(x):
    if isinstance(x,dict): return x
    if pd.isna(x): return {}
    try: return ast.literal_eval(x)
    except (ValueError,SyntaxError): return x

if RESUME_FROM_CHECKPOINT and ck_sel.exists():
    all_oof_rows=pd.read_csv(ck_oof).to_dict('records') if ck_oof.exists() else []
    all_trials=pd.read_csv(ck_trials).to_dict('records') if ck_trials.exists() else []
    selections=pd.read_csv(ck_sel).to_dict('records')
    selected_inner_tables=[pd.read_csv(ck_inner)] if ck_inner.exists() else []
    for row in all_trials+selections: row['candidate']=parse_candidate(row.get('candidate',{}))
    completed={(r['family'],int(r['seed']),int(r['outer_fold'])) for r in selections}
    print('Resumed',len(completed),'completed Variant-A outer tasks from',CHECKPOINT_DIR.resolve())
else:
    all_oof_rows=[]; all_trials=[]; selections=[]; selected_inner_tables=[]; completed=set()

def checkpoint():
    pd.DataFrame(all_oof_rows).drop_duplicates(['family','seed','fold','file'],keep='last').to_csv(ck_oof,index=False)
    pd.DataFrame(all_trials).drop_duplicates(['family','seed','outer_fold','trial'],keep='last').to_csv(ck_trials,index=False)
    pd.DataFrame(selections).drop_duplicates(['family','seed','outer_fold'],keep='last').to_csv(ck_sel,index=False)
    if selected_inner_tables:
        pd.concat(selected_inner_tables,ignore_index=True).drop_duplicates(
            ['family','seed','outer_fold','trial','inner_fold','file'],keep='last').to_csv(ck_inner,index=False)
    (CHECKPOINT_DIR/'checkpoint_manifest.json').write_text(json.dumps(dict(
        completed_tasks=[list(x) for x in sorted(completed)],
        n_recording_rows=len(all_oof_rows),saved_unix_time=time.time()),indent=2))

t0=time.time()
for seed in TRAIN_SEEDS:
    for k in range(N_FOLDS):
        key=('A',seed,k)
        if key in completed:
            print(f'[A] outer {k} seed {seed} skipped: checkpoint complete')
            continue
        best_idx,trials,inner=nested_search(SPACE_A,k,FOLDS[k]['train'],seed)
        selected=trials[best_idx]; cfg=selected['candidate']
        all_trials.extend(trials); selected_inner_tables.append(inner.assign(selected=True))
        p,best=fit_selected_outer(cfg,k,seed)
        all_oof_rows+=prediction_rows(FOLDS[k]['test'],p,family='A',seed=seed,fold=k)
        selections.append(dict(family='A',seed=seed,outer_fold=k,candidate=cfg,
            best_window_epoch=best['window'],best_recording_epoch=best['recording'],
            inner_participant_log_loss=selected['inner_participant_log_loss'],
            inner_participant_auc=selected['inner_participant_auc'],
            classifier_parameters=selected['classifier_parameters'],
            predictor_parameters=selected['predictor_parameters'],
            total_parameters=selected['total_parameters']))
        completed.add(key); checkpoint(); release_memory()
        print(f'[A] outer {k} seed {seed} SELECTED {cfg}')

expected={('A',seed,k) for seed in TRAIN_SEEDS for k in range(N_FOLDS)}
missing=expected-completed
if missing: raise RuntimeError(f'Variant-A experiment incomplete; missing tasks: {sorted(missing)}')
raw_oof=pd.DataFrame(all_oof_rows).drop_duplicates(['family','seed','fold','file'],keep='last')
trial_table=pd.DataFrame(all_trials).drop_duplicates(['family','seed','outer_fold','trial'],keep='last')
selection_table=pd.DataFrame(selections).drop_duplicates(['family','seed','outer_fold'],keep='last')
nested_oof=oof_frame(raw_oof.to_dict('records'))
inner_selected_oof=(pd.concat(selected_inner_tables,ignore_index=True)
                    if selected_inner_tables else pd.DataFrame())
if not inner_selected_oof.empty:
    inner_selected_oof=inner_selected_oof.drop_duplicates(
        ['family','seed','outer_fold','trial','inner_fold','file'],keep='last')
print(f'Completed/resumed Variant-A experiments in {(time.time()-t0)/3600:.2f} hours')
show_table(selection_table,'Selected Variant-A configuration independently inside every outer fold',limit=100)
show_table(trial_table,'All Variant-A inner-search trials; common inner splits within each outer fold',limit=60)


In [ ]:
# ============================================================
# Complete participant and recording OOF results
# ============================================================
nested_summary=summarize_oof(nested_oof,['family','seed'])
show_table(nested_summary.round(4),'Complete OOF metrics by model family and training seed')
nested_ci=pooled_bootstrap(nested_oof,['family','seed'])
show_table(nested_ci.round(4),'Participant OOF metrics and 95% stratified bootstrap intervals')

across_seed=(nested_summary.groupby('family')[[c for c in nested_summary.columns
             if c.startswith('participant_') and c!='participant_n']].agg(['mean','std']).round(4))
show_table(across_seed,'Training-seed variability on the same outer folds (SD is not participant CI)')

# Fold-level recording metrics are descriptive only.
nested_fold_rows=[]
for (family,seed,fold),g in nested_oof.groupby(['family','seed','fold']):
    nested_fold_rows.append(dict(family=family,seed=seed,fold=fold,
                                 **metric_dict(g['label'],g['probability'])))
nested_fold_metrics=pd.DataFrame(nested_fold_rows)
show_table(nested_fold_metrics.round(4),'Outer-fold recording metrics; training folds overlap, so these are not independent replicates')

plot_confusion_and_roc(nested_oof,['family','seed'],'family')


In [ ]:
# ============================================================
# Participant errors, HY severity, study patterns, and seed stability
# ============================================================
nested_participants,nested_errors,nested_pd_severity,nested_study=error_tables(nested_oof,['family','seed'])
show_table(nested_errors[['family','seed','fold','subject','study','diagnosis','hy_label','age',
                          'recordings','probability','prediction','confidence','nll']].round(4),
           'All misclassified outer-test participants',limit=250)
show_table(nested_pd_severity.round(4),
           'PD sensitivity by verified HY stage; small cells and Unknown are retained')
show_table(nested_study.round(4),'Participant OOF metrics by study and diagnosis')

seed_stability=(nested_participants.groupby(['family','subject']).agg(
    error_seeds=('correct',lambda x:int((x==0).sum())),n_seeds=('seed','nunique'),
    mean_probability=('probability','mean'),mean_nll=('nll','mean')).reset_index())
seed_stability=seed_stability.merge(nested_participants[['subject','study','diagnosis','hy_label','age']].drop_duplicates('subject'),
                                    on='subject',how='left')
show_table(seed_stability.sort_values(['error_seeds','mean_nll'],ascending=[False,False]).round(4),
           'Persistent versus seed-variable participant errors',limit=200)

if not inner_selected_oof.empty:
    inner_loss=participant_oof(inner_selected_oof,['family','seed','outer_fold','trial'])
    show_table(inner_loss.sort_values('nll',ascending=False)[['family','seed','outer_fold','subject',
               'diagnosis','hy_label','probability','prediction','nll']].head(40).round(4),
               'Highest-NLL selected inner-validation participants; repeated across outer folds, not independent people')


In [ ]:
# ============================================================
# Export complete Variant-A reproducibility and error-analysis tables
# ============================================================
outdir=CHECKPOINT_DIR; outdir.mkdir(exist_ok=True)
for name,df in [('recording_oof',nested_oof),('participant_oof',nested_participants),
                ('pooled_metrics',nested_summary),('participant_bootstrap',nested_ci),
                ('fold_metrics',nested_fold_metrics),('search_trials',trial_table),
                ('selected_configurations',selection_table),('participant_errors',nested_errors),
                ('pd_severity',nested_pd_severity),('study_metrics',nested_study),
                ('seed_stability',seed_stability),('selected_inner_oof',inner_selected_oof)]:
    df.to_csv(outdir/f'{name}.csv',index=False)
protocol=dict(model='Variant A: LP residual + squeeze-excitation + attention pooling',
              baseline_trained=False,gru_residual_trained=False,
              outer_folds=N_FOLDS,inner_folds=N_INNER,fold_seed=FOLD_SEED,
              training_seeds=TRAIN_SEEDS,trials_per_outer_fold=N_TRIALS,
              search_space=SPACE_A,selection_metric='pooled inner participant log loss',
              threshold=0.5,final_refit='all outer-training participants using inner-selected configuration and outer-training-selected epochs',
              data_pipeline='batch-generated official windows and sequence-wide wrap-padded recordings',
              restart_safe_checkpoints=True,tensorflow_version=tf.__version__,
              numpy_version=np.__version__,pandas_version=pd.__version__)
(outdir/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('Variant-A CSV and protocol outputs written to:',outdir.resolve())


## Reporting checklist and limitations

- Report all outer folds, all declared seeds, every selected Variant-A configuration, and the
  actual number of participants with sensor files. Do not choose the best seed or outer fold.
- Participant OOF metrics are primary; recording metrics are secondary. Repeated seeds do
  not create additional participants. Increase `TRAIN_SEEDS` for the final robustness run if
  compute permits, and declare every seed in advance.
- Bootstrap intervals are conditional on fitted OOF predictions and exclude model-refitting,
  new-split, external-site, and future-population uncertainty.
- This notebook does not train LPGNet or the GRU-residual model. Any comparison with notebook
  1 must use the corrected fold-local/inner-validation condition and aligned participant OOF
  predictions; availability of a baseline number alone does not establish statistical significance.
- Variant A's total parameter count includes the 198 LP coefficients. Squeeze-excitation and
  attention weights are model components, not automatic causal or clinical explanations.
- HY and study analyses are exploratory descriptions. `PD sensitivity` means correct detection
  of PD participants within a stage, not severity-classification accuracy.
- Checkpoints are for completing every declared task, not selecting favorable runs. Final
  metrics require all `TRAIN_SEEDS × 10` Variant-A outer tasks.
- Any change motivated by outer-test errors must be evaluated in a new nested experiment or,
  preferably, an external cohort rather than fed back into the reported folds.
